# UKB power cases: locus tracks (data processing)

From `31.UKB_Power_Cases` (`2026_output/UKB_power_cases/result/<unit>/`): six UKB blood-trait region × trait pairs where SuSiE-RSS with the sketch LD has less power than SuSiE-rRSS. For each, the saved inputs (`<unit>.inputs.rds`) and the three fits (`insample`, `susie_rss`, `susie_rrss`).

**Outputs:**
- `figure_data/UKB_Power_Cases.rds` (+ `.csv`): one row per unit × variant: position, z (sketch-harmonised, after the kriging fix), −log10 P, variant index (`idx`, position order), lead variant (max \|z\|), r with the lead in the reference sketch LD (`r_ref`, what the fits used) and in the in-sample LD (`r_insample`), and PIP / credible set (`0` = none) of each fit: `insample`, `rss` (SuSiE-RSS), `rrss` (SuSiE-rRSS pipeline result: SER model when flagged) and `rrss_full` (SuSiE-rRSS before fallback).
- `figure_data/UKB_Power_Cases_Summary.rds` (+ `.csv`): one row per unit: number of variants, n, lead, credible sets per fit, SER fallback.

In [3]:
source("../../R/utils.R")

data_name <- "UKB_Power_Cases"
root      <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_power_cases/result"
units     <- basename(list.dirs(root, recursive = FALSE))
units

[1] "chr11_122173839_123183181.Plateletcrit"   
[2] "chr11_15756615_17320581.Reticulocyte_perc"
[3] "chr13_49803063_51789698.Lymphocyte_perc"  
[4] "chr22_27931399_31198752.WBC_count"        
[5] "chr5_175313842_176069996.RDW"             
[6] "chr9_86078928_86867265.PDW"

In [4]:
## credible-set index per variant (0 = none)
cs_index <- function(fit, p) {
  cs <- rep(0L, p)
  for (k in seq_along(fit$sets$cs)) cs[fit$sets$cs[[k]]] <- k
  cs
}

one_unit <- function(u) {
  f    <- function(m) file.path(root, u, paste0(u, ".", m, ".rds"))
  inp  <- readRDS(f("inputs"))
  p    <- length(inp$z_sketch)
  lead <- which.max(abs(inp$z_sketch))
  stopifnot(length(inp$sumstat$pos) == p)

  rss  <- readRDS(f("susie_rss")); rrss <- readRDS(f("susie_rrss")); ins <- readRDS(f("insample"))
  for (o in list(rss, rrss, ins)) if (!is.null(o$error)) stop(u, " ", o$method, ": ", o$error)
  fits <- list(insample = ins$fit, rss = rss$fit, rrss = rrss$fit, rrss_full = rrss$full_fit)

  var <- tibble(unit = u, region = inp$region, trait = inp$trait, idx = seq_len(p), variant = inp$variant_ids,
                pos = inp$sumstat$pos, z = inp$z_sketch,
                logp = -(pnorm(-abs(inp$z_sketch), log.p = TRUE) + log(2)) / log(10),   # -log10(2*Phi(-|z|))
                lead = seq_len(p) == lead,
                r_ref = inp$sketch_LD[lead, ], r_insample = inp$insample_LD[lead, ])
  for (m in names(fits)) {
    stopifnot(length(fits[[m]]$pip) == p)
    var[[paste0("pip_", m)]] <- as.numeric(fits[[m]]$pip)
    var[[paste0("cs_", m)]]  <- cs_index(fits[[m]], p)
  }

  summ <- tibble(unit = u, region = inp$region, trait = inp$trait, p = p, n = inp$n,
                 lead_variant = inp$variant_ids[lead], lead_idx = lead, lead_pos = var$pos[lead], max_logp = max(var$logp),
                 n_cs_insample = length(ins$fit$sets$cs), n_cs_rss = length(rss$fit$sets$cs),
                 n_cs_rrss = length(rrss$fit$sets$cs), n_cs_rrss_full = length(rrss$full_fit$sets$cs),
                 ser_fallback = isTRUE(rrss$ser_fallback))
  list(var = var, summ = summ)
}

res  <- map(units, one_unit)
var  <- map_dfr(res, "var")
summ <- map_dfr(res, "summ")
summ %>% as.data.frame()

save_data(var,  data_name)
save_data(summ, paste0(data_name, "_Summary"))

unit,region,trait,p,n,lead_variant,lead_idx,lead_pos,max_logp,n_cs_insample,n_cs_rss,n_cs_rrss,n_cs_rrss_full,ser_fallback
<chr>,<chr>,<chr>,<int>,<int>,<chr>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,<lgl>
chr11_122173839_123183181.Plateletcrit,chr11:122173839-123183181,Plateletcrit,3990,248980,chr11:122553131:A:G,780,122553131,9.048528,2,1,2,2,FALSE
chr11_15756615_17320581.Reticulocyte_perc,chr11:15756615-17320581,Reticulocyte_perc,4622,248980,chr11:16613877:G:C,2479,16613877,20.015359,3,3,2,2,FALSE
chr13_49803063_51789698.Lymphocyte_perc,chr13:49803063-51789698,Lymphocyte_perc,6883,248980,chr13:50047287:A:G,2965,50047287,9.469705,2,1,2,2,FALSE
chr22_27931399_31198752.WBC_count,chr22:27931399-31198752,WBC_count,10756,248980,chr22:29054205:A:G,4808,29054205,9.810121,4,2,4,4,FALSE
chr5_175313842_176069996.RDW,chr5:175313842-176069996,RDW,1952,248980,chr5:176136839:A:G,374,176136839,20.903425,1,7,1,1,FALSE
chr9_86078928_86867265.PDW,chr9:86078928-86867265,PDW,2613,248980,chr9:83980583:A:G,1589,83980583,9.338418,1,5,1,5,TRUE
